In [ ]:
# RegMap – Sentence‑BERT Fine‑Tuning for NIST‑to‑HIPAA Mapping

In [8]:
import pandas as pd
import numpy as np
import random
import json
import datetime
from pathlib import Path

from sklearn.model_selection import train_test_split

# sentence-transformers
from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation
)
from torch.utils.data import DataLoader
import torch

# ---------- Paths ----------
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path('../models/regmap-embedder')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Reproducibility ----------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [9]:
# Load the official NIST-HIPAA crosswalk CSV
crosswalk_raw = pd.read_csv(DATA_RAW / 'nist_800_53_rev5_hipaa_crosswalk.csv')

# Clean column names (remove newlines, collapse spaces)
crosswalk_raw.columns = (
    crosswalk_raw.columns
    .str.replace('\n', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Rename to our standard names
crosswalk = crosswalk_raw.rename(columns={
    'Focal Document Element': 'nist_control_id',
    'Focal Document Element Description': 'nist_text',
    'Reference Document Element': 'hipaa_citation',
    'Reference Document Element Description': 'hipaa_text'
})

# Keep only the columns we need
crosswalk = crosswalk[['nist_control_id', 'nist_text', 'hipaa_citation', 'hipaa_text']]

# Drop rows where critical text is missing
crosswalk.dropna(subset=['nist_text', 'hipaa_text', 'hipaa_citation'], inplace=True)

# Clean whitespace
for col in ['nist_text', 'hipaa_text', 'hipaa_citation']:
    crosswalk[col] = crosswalk[col].astype(str).str.strip()

# Remove exact duplicate pairs
crosswalk = crosswalk.drop_duplicates(subset=['nist_control_id', 'hipaa_citation'])

# All rows in this crosswalk are valid mappings → label = 1
crosswalk['label'] = 1

print(f"Positive pairs (true mappings): {len(crosswalk)}")
crosswalk.head()

Positive pairs (true mappings): 40


,nist_control_id,nist_text,hipaa_citation,hipaa_text,label
0,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3),Workforce Security: Implement policies and pro...,1
1,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(3)(ii)(A),Authorization and/or Supervision (A): Implemen...,1
2,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4),Information Access Management: Implement polic...,1
3,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(B),Access Authorization (A): Implement policies a...,1
4,AC-01,"a. Develop, document, and disseminate to [Assi...",164.308(a)(4)(ii)(C),Access Establishment and Modification (A): Imp...,1


In [5]:
# Extract unique IDs and citations
nist_ids = crosswalk['nist_control_id'].unique()
hipaa_citations = crosswalk['hipaa_citation'].unique()

# Build a dictionary: nist_id -> set of positive hipaa citations
positive_map = {}
for _, row in crosswalk.iterrows():
    positive_map.setdefault(row['nist_control_id'], set()).add(row['hipaa_citation'])

# Generate negative pairs (random NIST + HIPAA not in positive_map)
num_negatives_target = len(crosswalk) * 3   # 3 negatives per positive
negative_pairs = []
attempts = 0
max_attempts = num_negatives_target * 10     # safety valve

while len(negative_pairs) < num_negatives_target and attempts < max_attempts:
    nist_id = random.choice(nist_ids)
    neg_hipaa = random.choice(hipaa_citations)
    if neg_hipaa not in positive_map.get(nist_id, set()):
        negative_pairs.append({
            'nist_control_id': nist_id,
            'hipaa_citation': neg_hipaa,
            'label': 0
        })
    attempts += 1

print(f"Negative pairs generated: {len(negative_pairs)}")

# Convert to DataFrame
neg_df = pd.DataFrame(negative_pairs)

# Merge with the full text of the NIST control and HIPAA citation
nist_texts = crosswalk[['nist_control_id', 'nist_text']].drop_duplicates()
hipaa_texts = crosswalk[['hipaa_citation', 'hipaa_text']].drop_duplicates()
neg_df = neg_df.merge(nist_texts, on='nist_control_id', how='inner')
neg_df = neg_df.merge(hipaa_texts, on='hipaa_citation', how='inner')

# Combine positives and negatives, then shuffle
full_df = pd.concat([crosswalk, neg_df], ignore_index=True)
full_df = full_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Full training dataset size: {len(full_df)}")
print("Label distribution:")
print(full_df['label'].value_counts())

Negative pairs generated: 120
Full training dataset size: 160
Label distribution:
label
0    120
1     40
Name: count, dtype: int64


In [6]:
full_df.to_csv(DATA_PROCESSED / 'training_pairs.csv', index=False)
print("Saved training_pairs.csv")

Saved training_pairs.csv


In [ ]:
## Train / Validation / Test Split

In [7]:
# Stratified split to keep label balance
train_val, test = train_test_split(
    full_df,
    test_size=0.15,
    random_state=SEED,
    stratify=full_df['label']
)
train, val = train_test_split(
    train_val,
    test_size=0.15,   # 15% of remaining → ~12.75% of total
    random_state=SEED,
    stratify=train_val['label']
)

print(f"Train:   {len(train)}  (label 1 = {train['label'].sum()})")
print(f"Val:     {len(val)}    (label 1 = {val['label'].sum()})")
print(f"Test:    {len(test)}   (label 1 = {test['label'].sum()})")

Train:   115  (label 1 = 29)
Val:     21    (label 1 = 5)
Test:    24   (label 1 = 6)


In [6]:
def df_to_examples(df):
    examples = []
    for _, row in df.iterrows():
        examples.append(
            InputExample(
                texts=[row['nist_text'], row['hipaa_text']],
                label=float(row['label'])
            )
        )
    return examples

train_examples = df_to_examples(train)
val_examples   = df_to_examples(val)
test_examples  = df_to_examples(test)

print(f"Train examples: {len(train_examples)}")
print(f"Val examples:   {len(val_examples)}")
print(f"Test examples:  {len(test_examples)}")

Train examples: 115
Val examples:   21
Test examples:  24


In [13]:
# Check which variable we are about to split
try:
    print("full_df shape:", full_df.shape)
    print("full_df label counts:")
    print(full_df['label'].value_counts())
except NameError:
    print("full_df does not exist!")

try:
    print("labeled shape:", labeled.shape)
    print("labeled label counts:")
    print(labeled['label'].value_counts())
except NameError:
    print("labeled does not exist.")

full_df shape: (160, 5)
full_df label counts:
label
0    120
1     40
Name: count, dtype: int64
labeled does not exist.


In [ ]:
## Model, Loss Function, and Validation Evaluator

In [10]:
# Fast, lightweight transformer that works well on CPU
model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = losses.ContrastiveLoss(model=model, margin=0.5)
print("Base model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Base model loaded.


In [11]:
# Build a corpus of unique HIPAA texts from the validation set
val_hipaa_texts = val['hipaa_text'].unique()
corpus = {i: text for i, text in enumerate(val_hipaa_texts)}
corpus_list = list(corpus.values())

# Map each NIST query to the HIPAA document(s) that are truly relevant
queries = {}
relevant_docs = {}
qid = 0
for _, row in val.iterrows():
    # Only include positive queries for evaluation
    if row['label'] == 1:
        # Find the corpus id of the true HIPAA text
        true_text = row['hipaa_text']
        true_id = None
        for idx, ctext in corpus.items():
            if ctext == true_text:
                true_id = idx
                break
        if true_id is not None:
            queries[qid] = row['nist_text']
            relevant_docs[qid] = {true_id}
            qid += 1

print(f"Validation queries with positive mappings: {len(queries)}")

# Create the evaluator
ir_evaluator = evaluation.InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="val-hipaa",
    show_progress_bar=True
)

Validation queries with positive mappings: 5


In [12]:
BATCH_SIZE = 8
EPOCHS = 10
WARMUP_STEPS = int(len(train_examples) / BATCH_SIZE * 0.1)

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    evaluator=ir_evaluator,
    evaluation_steps=500,
    output_path=str(MODEL_DIR),
    save_best_model=True,
    show_progress_bar=True
)

NameError: name 'train_examples' is not defined

In [ ]:
## Training (manual PyTorch loop)

In [12]:
# from torch.optim import AdamW

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# # Use the model's own collate function to turn InputExamples into features
# train_dataloader = DataLoader(
#     train_examples,
#     shuffle=True,
#     batch_size=BATCH_SIZE,
#     collate_fn=model.smart_batching_collate
# )

# optimizer = AdamW(model.parameters(), lr=2e-5)
# scheduler = torch.optim.lr_scheduler.LinearLR(
#     optimizer, start_factor=1.0, end_factor=0.1, total_iters=max(1, WARMUP_STEPS)
# )

# num_epochs = EPOCHS
# best_score = 0
# best_model_path = MODEL_DIR / "best"
# best_model_path.mkdir(parents=True, exist_ok=True)

# for epoch in range(1, num_epochs + 1):
#     model.train()
#     total_loss = 0
#     for features, labels in train_dataloader:
#         # features is a list of dicts (input_ids, attention_mask, …), labels is a tensor
#         loss_value = train_loss(features, labels)
#         optimizer.zero_grad()
#         loss_value.backward()
#         optimizer.step()
#         scheduler.step()
#         total_loss += loss_value.item()

#     avg_loss = total_loss / len(train_dataloader)
#     print(f"Epoch {epoch}/{num_epochs} - Loss: {avg_loss:.4f}")

#     # --- Evaluate on validation set ---
#     model.eval()
#     eval_output = ir_evaluator(model, output_path=None)
#     # ir_evaluator returns a dict with keys like 'val-hipaa_cosine_recall@5'
#     # Print all keys to see exact metric names
#     print("Eval metrics:", eval_output)
#     score = eval_output.get('val-hipaa_cosine_recall@5', eval_output.get('val-hipaa_recall@5', 0.0))
#     print(f"  Recall@5: {score:.4f}")

#     if score > best_score:
#         best_score = score
#         model.save(str(best_model_path))
#         print(f"  ✓ New best model saved")

# # Load the best model for final testing
# model = SentenceTransformer(str(best_model_path))
# print(f"\nBest validation Recall@5: {best_score:.4f}")

Epoch 1/10 - Loss: 0.0158


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  3.89it/s]


Eval metrics: {'val-hipaa_cosine_accuracy@1': 0.0, 'val-hipaa_cosine_accuracy@3': 0.2, 'val-hipaa_cosine_accuracy@5': 0.4, 'val-hipaa_cosine_accuracy@10': 0.8, 'val-hipaa_cosine_precision@1': 0.0, 'val-hipaa_cosine_precision@3': 0.06666666666666667, 'val-hipaa_cosine_precision@5': 0.08, 'val-hipaa_cosine_precision@10': 0.08, 'val-hipaa_cosine_recall@1': 0.0, 'val-hipaa_cosine_recall@3': 0.2, 'val-hipaa_cosine_recall@5': 0.4, 'val-hipaa_cosine_recall@10': 0.8, 'val-hipaa_cosine_ndcg@10': 0.30415427601105244, 'val-hipaa_cosine_mrr@10': 0.15888888888888889, 'val-hipaa_cosine_map@100': 0.17555555555555552}
  Recall@5: 0.4000


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Best validation Recall@5: 0.8000


In [13]:
# # Build test corpus
# test_hipaa_texts = test['hipaa_text'].unique()
# test_corpus = {i: text for i, text in enumerate(test_hipaa_texts)}
# test_corpus_list = list(test_corpus.values())
# corpus_embeddings = model.encode(test_corpus_list, convert_to_tensor=True)

# recall_1 = recall_3 = recall_5 = mrr = 0
# total_queries = 0

# for _, row in test.iterrows():
#     if row['label'] != 1:
#         continue
#     query_text = row['nist_text']
#     true_text = row['hipaa_text']

#     # Find corpus id of true HIPAA
#     true_id = None
#     for idx, ctext in test_corpus.items():
#         if ctext == true_text:
#             true_id = idx
#             break
#     if true_id is None:
#         continue

#     query_embed = model.encode(query_text, convert_to_tensor=True)
#     cos_scores = torch.nn.functional.cosine_similarity(query_embed, corpus_embeddings)
#     sorted_indices = torch.argsort(cos_scores, descending=True)
#     rank = (sorted_indices == true_id).nonzero(as_tuple=True)[0].item() + 1

#     if rank <= 1:
#         recall_1 += 1
#     if rank <= 3:
#         recall_3 += 1
#     if rank <= 5:
#         recall_5 += 1
#     mrr += 1 / rank
#     total_queries += 1

# recall_1 /= total_queries
# recall_3 /= total_queries
# recall_5 /= total_queries
# mrr /= total_queries

# print(f"Test queries evaluated: {total_queries}")
# print(f"Recall@1: {recall_1:.4f}")
# print(f"Recall@3: {recall_3:.4f}")
# print(f"Recall@5: {recall_5:.4f}")
# print(f"MRR:      {mrr:.4f}")

Test queries evaluated: 6
Recall@1: 0.1667
Recall@3: 0.8333
Recall@5: 0.8333
MRR:      0.4960


In [14]:
# results = {
#     'model': 'all-MiniLM-L6-v2 fine-tuned on NIST-HIPAA crosswalk (manual training)',
#     'positive_pairs': len(crosswalk),
#     'negative_pairs': len(neg_df),
#     'recall@1': recall_1,
#     'recall@3': recall_3,
#     'recall@5': recall_5,
#     'mrr': mrr,
#     'date': str(datetime.date.today())
# }

# with open(MODEL_DIR / 'evaluation_results.json', 'w') as f:
#     json.dump(results, f, indent=2)

# print("Saved evaluation_results.json")
# print("✅ RegMap embedder training complete.")

Saved evaluation_results.json
✅ RegMap embedder training complete.
